In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [2]:
import json

basic_path = '/home/tts26/huyng/MORAI/SSS/results/treebench_qwen2.5-vl-7b-instruct_basic.json'
grit_path = '/home/tts26/huyng/MORAI/SSS/results/treebench_qwen2.5-vl-7b-instruct_grit.json'

with open(basic_path, 'r') as f:
    basic = json.load(f)

with open(grit_path, 'r') as f:
    grit = json.load(f)

basic_details = basic.get('details', [])
grit_details = grit.get('details', [])

# Create a mapping of question_id or image to the details if they are lists
def get_mapping(details_list):
    # Depending on how details are structured, try to map by a unique identifier
    mapping = {}
    for item in details_list:
        # We assume there might be a 'question_id' or 'id' or we can just use the index if they are in the exact same order.
        # Let's check keys of the first item
        if 'question_id' in item:
            mapping[item['question_id']] = item
        elif 'id' in item:
            mapping[item['id']] = item
        else:
            # use image path + question as key maybe? or just assume same order?
            # if no ID, we will just zip them if lengths match
            pass
    return mapping

if isinstance(basic_details, list) and isinstance(grit_details, list):
    if len(basic_details) > 0 and 'question_id' in basic_details[0]:
        b_map = {item['question_id']: item for item in basic_details}
        g_map = {item['question_id']: item for item in grit_details}
        
        changed_to_wrong = []
        for qid, b_item in b_map.items():
            if qid in g_map:
                g_item = g_map[qid]
                # Assuming 'is_correct' or similar field exists
                # Or 'pred_ans' == 'ground_truth'
                b_correct = b_item.get('is_correct', b_item.get('pred_ans') == b_item.get('ground_truth'))
                g_correct = g_item.get('is_correct', g_item.get('pred_ans') == g_item.get('ground_truth'))
                
                if b_correct and not g_correct:
                    changed_to_wrong.append((qid, b_item, g_item))
                    
        print(f"Total questions where GRIT changed from Correct to Wrong: {len(changed_to_wrong)}")
        for i, (qid, b_item, g_item) in enumerate(changed_to_wrong[:3]): # print 3 examples
            print(f"\n--- Example {i+1} ---")
            print(f"Question ID: {qid}")
            print(f"Category: {b_item.get('category')}")
            print(f"Ground Truth: {b_item.get('ground_truth')}")
            print(f"Basic Prediction: {b_item.get('pred_ans')} (Correct: {b_correct})")
            print(f"GRIT Prediction: {g_item.get('pred_ans')} (Correct: {g_correct})")
            print(f"Basic Text Output:\n{b_item.get('prediction_text', b_item.get('text'))[:500]}...")
            print(f"GRIT Text Output:\n{g_item.get('prediction_text', g_item.get('text'))[:500]}...")
            if 'grit_invocations' in g_item:
                print(f"GRIT Invocations: {g_item['grit_invocations']}")
    else:
        # zip if no ID
        changed_to_wrong = []
        for b_item, g_item in zip(basic_details, grit_details):
            b_correct = b_item.get('is_correct', b_item.get('pred_ans') == b_item.get('ground_truth'))
            g_correct = g_item.get('is_correct', g_item.get('pred_ans') == g_item.get('ground_truth'))
            
            if b_correct and not g_correct:
                changed_to_wrong.append((b_item, g_item))
                
        print(f"Total questions where GRIT changed from Correct to Wrong: {len(changed_to_wrong)}")
        for i, (b_item, g_item) in enumerate(changed_to_wrong[:3]):
            print(f"\n--- Example {i+1} ---")
            print(f"Category: {b_item.get('category')}")
            print(f"Question: {b_item.get('question')}")
            print(f"Ground Truth: {b_item.get('ground_truth')}")
            print(f"Basic Prediction: {b_item.get('pred_ans')}")
            print(f"GRIT Prediction: {g_item.get('pred_ans')}")
            print(f"Basic Text: {b_item.get('prediction_text', b_item.get('output'))[:500]}")
            print(f"GRIT Text: {g_item.get('prediction_text', g_item.get('output'))[:500]}")


Total questions where GRIT changed from Correct to Wrong: 45

--- Example 1 ---
Category: Reasoning/Perspective Transform
Question: From the perspective of the red car, in which direction is the man walking a dog on the left side of the image located?
Ground Truth: B
Basic Prediction: None
GRIT Prediction: None
Basic Text: Based on the image, the red car is in the center of the image, and the man walking a dog is to the left of the red car. The man is walking in the direction of the road, which is to the right of the red car. Therefore, the man is in the front right direction relative to the red car.

</think>
<answer>B</answer>
GRIT Text: From the perspective of the red car, the man walking a dog on the left side of the image is located in the front left direction. The man is walking away from the red car, and the dog is in front of the man, indicating that the man is in the front left direction relative to the red car.

</think>
<answer>A</answer>

--- Example 2 ---
Category: Reasoni

In [3]:
import json

basic_path = '/home/tts26/huyng/MORAI/SSS/results/treebench_qwen2.5-vl-7b-instruct_basic.json'
grit_path = '/home/tts26/huyng/MORAI/SSS/results/treebench_qwen2.5-vl-7b-instruct_grit.json'

with open(basic_path, 'r') as f:
    basic = json.load(f)

with open(grit_path, 'r') as f:
    grit = json.load(f)

basic_details = basic.get('details', [])
grit_details = grit.get('details', [])

changed_to_wrong = []
for b_item, g_item in zip(basic_details, grit_details):
    b_correct = b_item.get('is_correct', b_item.get('pred_ans') == b_item.get('ground_truth'))
    g_correct = g_item.get('is_correct', g_item.get('pred_ans') == g_item.get('ground_truth'))
    
    if b_correct and not g_correct:
        changed_to_wrong.append((b_item, g_item))

categories_to_check = ['Perception/Physical State', 'Perception/OCR']

for cat in categories_to_check:
    print(f"\n==== Category: {cat} ====")
    cat_items = [x for x in changed_to_wrong if x[0].get('category') == cat]
    for i, (b_item, g_item) in enumerate(cat_items[:2]):
        print(f"\n--- Example {i+1} ---")
        print(f"Question: {b_item.get('question')}")
        print(f"Ground Truth: {b_item.get('ground_truth')}")
        print(f"Basic Text:\n{b_item.get('prediction_text', b_item.get('text'))[:300]}...")
        print(f"GRIT Text:\n{g_item.get('prediction_text', g_item.get('text'))[:300]}...")



==== Category: Perception/Physical State ====

--- Example 1 ---
Question: What is the condition of the rear cargo door on the small white box-shaped truck parked in the leftmost lane beside the row of orange traffic cones?
Ground Truth: A
Basic Text:
Upon analyzing the image, the small white box-shaped truck parked in the leftmost lane beside the row of orange traffic cones appears to have its rear cargo door in a closed and latched position. There are no visible signs of the door being open or missing, and the door appears to be in a standard c...
GRIT Text:
Based on the image, the small white box-shaped truck parked in the leftmost lane beside the row of orange traffic cones appears to be a standard delivery truck. The rear cargo door is not visible in the image, which makes it impossible to determine its condition. However, the truck is parked, and th...

--- Example 2 ---
Question: What is the condition of the top section of the tall, dark blue skyscraper located to the immediate